In [ ]:
# =============================================================
# IMPORTS
# =============================================================
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.decomposition import PCA
from matplotlib.animation import FuncAnimation

In [ ]:
# =============================================================
# DATASET
# =============================================================

# MNIST: 60000 train / 10000 test изображений 28x28

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

In [ ]:
# нормализация в [0,1]
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

# преобразование изображений в вектор
# автоэнкодеры работают с вектором признаков
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# размер латентного пространства
latent_dim = 2

In [ ]:
# =============================================================
# PCA
# =============================================================

# линейное снижение размерности для сравнения
pca = PCA(n_components=2)

z_pca = pca.fit_transform(x_test)

plt.figure(figsize=(6, 5))
plt.scatter(z_pca[:, 0], z_pca[:, 1], c=y_test, cmap="tab10", s=5)
plt.title("PCA projection")
plt.colorbar()
plt.show()

In [ ]:
# =============================================================
# AUTOENCODER
# =============================================================

# encoder: x -> z

inputs = keras.Input(shape=(784,))
x = layers.Dense(256, activation="relu")(inputs)
z = layers.Dense(latent_dim)(x)

encoder_ae = keras.Model(inputs, z)

# decoder: z -> x_hat

latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(256, activation="relu")(latent_inputs)
outputs = layers.Dense(784, activation="sigmoid")(x)

decoder_ae = keras.Model(latent_inputs, outputs)

# полная модель

outputs = decoder_ae(encoder_ae(inputs))
autoencoder = keras.Model(inputs, outputs)

# стандартная reconstruction loss

autoencoder.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

In [ ]:
autoencoder.fit(
    x_train,
    x_train,
    epochs=15,
    batch_size=128,
    validation_data=(x_test, x_test)
)

In [ ]:
# =============================================================
# VAE
# =============================================================

# слой с reparameterization trick

class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs

        # ε ~ N(0,1)
        epsilon = tf.random.normal(shape=tf.shape(z_mean))

        # z = μ + σ * ε
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
# -----------------------------
# Encoder VAE
# -----------------------------

encoder_inputs = keras.Input(shape=(784,))
x = layers.Dense(256, activation="relu")(encoder_inputs)

# encoder предсказывает параметры распределения
z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

# sampling из распределения
z = Sampling()([z_mean, z_log_var])

encoder_vae = keras.Model(
    encoder_inputs,
    [z_mean, z_log_var, z]
)

In [ ]:
# -----------------------------
# Decoder VAE
# -----------------------------

latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(256, activation="relu")(latent_inputs)

# вероятность пикселя
decoder_outputs = layers.Dense(784, activation="sigmoid")(x)

decoder_vae = keras.Model(latent_inputs, decoder_outputs)

In [ ]:
# =============================================================
# VAE MODEL
# =============================================================

# BinaryCrossentropy без автоматической редукции
bce = keras.losses.BinaryCrossentropy(reduction="none")


class VAE(keras.Model):

    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:
            # encoder
            z_mean, z_log_var, z = self.encoder(data)

            # decoder
            reconstruction = self.decoder(z)

            # reconstruction loss
            rec_loss = bce(data, reconstruction)

            # суммируем по пикселям
            rec_loss = tf.reduce_mean(tf.reduce_sum(rec_loss, axis=-1))

            # KL divergence
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var
                    - tf.square(z_mean)
                    - tf.exp(z_log_var),
                    axis=1
                )
            )

            loss = rec_loss + kl_loss

        grads = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        return {
            "loss": loss,
            "reconstruction_loss": rec_loss,
            "kl_loss": kl_loss
        }

In [ ]:
vae = VAE(encoder_vae, decoder_vae)

vae.compile(
    optimizer=keras.optimizers.Adam()
)

In [ ]:
vae.fit(
    x_train,
    epochs=15,
    batch_size=128
)

In [ ]:
# =============================================================
# LATENT SPACE
# =============================================================

# AE
z_ae = encoder_ae.predict(x_test)

# VAE (используем mean)
z_mean, _, _ = encoder_vae.predict(x_test)

In [ ]:
# =============================================================
# COMPARISON
# =============================================================

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.scatter(z_pca[:, 0], z_pca[:, 1], c=y_test, cmap="tab10", s=5)
plt.title("PCA")

plt.subplot(1, 3, 2)
plt.scatter(z_ae[:, 0], z_ae[:, 1], c=y_test, cmap="tab10", s=5)
plt.title("Autoencoder")

plt.subplot(1, 3, 3)
plt.scatter(z_mean[:, 0], z_mean[:, 1], c=y_test, cmap="tab10", s=5)
plt.title("Variational Autoencoder")

plt.show()

In [ ]:
# =============================================================
# GENERATION GRID AE
# =============================================================

grid_size = 20

grid_x = np.linspace(-3, 3, grid_size)
grid_y = np.linspace(-3, 3, grid_size)

figure_ae = np.zeros((28 * grid_size, 28 * grid_size))

for i, yi in enumerate(grid_y):
    for j, xi in enumerate(grid_x):
        z_sample = np.array([[xi, yi]])

        # генерация из latent space
        x_decoded = decoder_ae.predict(z_sample, verbose=0)

        digit = x_decoded.reshape(28, 28)

        figure_ae[
        i * 28:(i + 1) * 28,
        j * 28:(j + 1) * 28
        ] = digit

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(figure_ae, cmap="gray")
plt.title("AE latent space generation")
plt.show()

In [ ]:
grid_x

In [ ]:
# =============================================================
# GENERATION GRID VAE
# =============================================================

grid_size = 20

grid_x = np.linspace(-3, 3, grid_size)
grid_y = np.linspace(-3, 3, grid_size)

figure_vae = np.zeros((28 * grid_size, 28 * grid_size))

for i, yi in enumerate(grid_y):
    for j, xi in enumerate(grid_x):
        z_sample = np.array([[xi, yi]])

        # генерация из latent space
        x_decoded = decoder_vae.predict(z_sample, verbose=0)

        digit = x_decoded.reshape(28, 28)

        figure_vae[
        i * 28:(i + 1) * 28,
        j * 28:(j + 1) * 28
        ] = digit

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(figure_vae, cmap="gray")
plt.title("VAE latent space generation")
plt.show()

In [ ]:
fig, axis = plt.subplots(1,2, figsize=(16,8))
axis[0].imshow(figure_ae, cmap="gray")
axis[0].axis("off")
axis[0].set_title("AE")

axis[1].imshow(figure_vae, cmap="gray")
axis[1].axis("off")
axis[1].set_title("VAE")

plt.tight_layout()
plt.show()

In [ ]:
z_start = z_mean[0]
z_end = z_mean[1]

In [ ]:
# =============================================================
# COMPARISON
# =============================================================
%matplotlib notebook
plt.figure(figsize=(5, 5))

z_line = np.array([(1 - alpha) * z_start + alpha * z_end for alpha in np.linspace(0,1,100)])

plt.scatter(z_mean[:, 0], z_mean[:, 1], c=y_test, cmap="tab10", s=5)
plt.colorbar()
plt.scatter(z_line[:,0], z_line[:,1], c="black", linewidth=1)
plt.title("Variational Autoencoder")
plt.show()

In [ ]:
# =============================================================
# LATENT INTERPOLATION
# =============================================================

z_start = z_mean[0]
z_end = z_mean[1]
frames = 60
fig, ax = plt.subplots()
img = ax.imshow(np.zeros((28, 28)), cmap="gray", vmin=0, vmax=1)
ax.axis("off")


def update(frame):
    alpha = frame / frames
    # линейная интерполяция
    z = (1 - alpha) * z_start + alpha * z_end
    decoded = decoder_vae.predict(
        z.reshape(1, 2),
        verbose=0
    )
    digit = decoded.reshape(28, 28)
    img.set_data(digit)
    return [img]


anim = FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=80,
    repeat=True
)

plt.show()